In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

### Loading Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

In [ ]:
train.drop(['Id'], axis = 1, inplace = True)
test.drop(['Id'], axis = 1, inplace = True)

#### Seperating Numerical and Categorical values

In [ ]:
num_cols = train.select_dtypes(include = 'number').columns.tolist()
cat_cols = train.select_dtypes(include = 'object').columns.tolist()

#### Handling Numerical Columns

In [ ]:
num_cols

In [ ]:
# Checking for null values by %age
null_num = train[num_cols].isna().sum()

null_num = null_num.to_frame(name = 'null_counts')
null_num = null_num[null_num['null_counts'] > 0]
null_num['percentage'] = (null_num['null_counts'] / len(train)) * 100
null_num

In [ ]:
# dropping columns where null values is > 50%

num_drop = null_num[null_num['percentage'] > 50]
num_drop = num_drop.index
num_drop

In [ ]:
train.drop(columns = num_drop, inplace = True)
test.drop(columns = num_drop, inplace = True)

In [ ]:
train[num_cols].head()

In [ ]:
for i in num_cols:
    print(f'{i}: {train[i].unique()}')

In [ ]:
num_pipeline = Pipeline(steps = [
    ('Impute', SimpleImputer(strategy = 'mean')),
    ('Scalar', StandardScaler())
])

#### Handling Categorical Columns

In [ ]:
cat_cols

In [ ]:
cat_null = train[cat_cols].isna().sum()
cat_null = cat_null.to_frame(name = 'null_counts')
cat_null = cat_null[cat_null['null_counts'] > 0]
cat_null['percentage'] = (cat_null['null_counts'] / len(train)) * 100
cat_null

In [ ]:
# Dropping columns where null values > 50%

cat_drop = cat_null[cat_null['percentage'] > 50]
print(cat_drop.index)
cat_drop = cat_drop.index

In [ ]:
train.drop(columns = cat_drop, inplace = True)
test.drop(columns = cat_drop, inplace = True)

In [ ]:
# remove the columns we dropped from the list of categorical columns;
# doing a membership test avoids the ambiguous-array comparison.
cat_cols = [c for c in cat_cols if c not in cat_drop]

In [ ]:
train[cat_cols].head()

In [ ]:
for i in cat_cols:
    print(f'{i}: {train[i].unique()}')
    print(f'{i}: {train[i].value_counts()}')

In [ ]:
#Handling Ordinal Values
lShape = ['IR3', 'IR2', 'IR1', 'Reg']
lSlope = ['Sev', 'Mod', 'Gtl']

In [ ]:
neighborhood_median = train.groupby('Neighborhood')['SalePrice'].median()
train['Neighborhood'] = train['Neighborhood'].map(neighborhood_median)
test['Neighborhood'] = test['Neighborhood'].map(neighborhood_median).fillna(neighborhood_median.median())

In [ ]:
# as street column is highly
train.drop(['Street'], axis = 1, inplace = True)
test.drop(['Street'], axis = 1, inplace = True)
cat_cols.remove('Street')

In [ ]:
train.drop(['Utilities'], axis = 1, inplace = True)
test.drop(['Utilities'], axis = 1, inplace = True)
cat_cols.remove('Utilities')

In [ ]:
(train['Condition2'] != 'Norm').sum()

In [ ]:
# as condition2 is highly biased we'll drop it
train.drop(['Condition2'], axis = 1, inplace = True)
test.drop(['Condition2'], axis = 1, inplace = True)
cat_cols.remove('Condition2')

In [ ]:
(train['RoofMatl'] != 'CompShg').sum()

In [ ]:
# RoofMatl is highly biased column so we'll drop it
train.drop(['RoofMatl'], axis = 1, inplace = True)
test.drop(['RoofMatl'], axis = 1, inplace = True)
cat_cols.remove('RoofMatl')

In [ ]:
print(train['Exterior1st'].nunique())
print(train['Exterior2nd'].nunique())

print((train['Exterior1st'] == train['Exterior2nd']).sum())

In [ ]:
exterior_median = train.groupby('Exterior1st')['SalePrice'].median()
train['Exterior1st'] = train['Exterior1st'].map(exterior_median)
test['Exterior1st'] = test['Exterior1st'].map(exterior_median).fillna(exterior_median.median())

#dropping exterior 2nd as it is same as exterior 1st and will create noise for the model
train.drop(['Exterior2nd'], axis = 1, inplace = True)
test.drop(['Exterior2nd'], axis = 1, inplace = True)
cat_cols.remove('Exterior2nd')

In [ ]:
QualOrder = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
bsmt_quality_order = ['None', 'No', 'Mn', 'Av', 'Gd']
bsmt_fin_order = ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
Electrical_order = ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr']
Functional_order = ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ']
Garage_Qual = ['None', 'Unf', 'RFn', 'Fin']
Paved_drive = ['N', 'P', 'Y']

In [ ]:
train['Heating'] = train['Heating'].apply(
    lambda x: x if x == 'GasA' else 'Other'
)
test['Heating'] = test['Heating'].apply(
    lambda x: x if x == 'GasA' else 'Other'
)

In [ ]:
saletype_median = train.groupby('SaleType')['SalePrice'].median()
train['SaleType'] = train['SaleType'].map(saletype_median)
test['SaleType'] = test['SaleType'].map(saletype_median).fillna(saletype_median.median())

In [ ]:
ordinal_pipeline = Pipeline(steps = [
    ('Impute', SimpleImputer(strategy = 'constant', fill_value = 'None')),
    ('Encoder', OrdinalEncoder(categories = [
        lShape, #LotShape
        lSlope, #LotSlope
        QualOrder, #ExterQual
        QualOrder, #ExrterCond
        QualOrder, #BsmtQual
        QualOrder, #BsmtCond
        bsmt_quality_order, #BasementExposure
        bsmt_fin_order, #BsmtFinType1
        bsmt_fin_order, #BsmtFinType2
        QualOrder, #HeatingQc
        Electrical_order, #Electrical
        QualOrder, #KitchenQual
        Functional_order, #Functional
        QualOrder, #FireplaceQu
        Garage_Qual, #GarageFinish
        QualOrder, #GarageQual
        QualOrder, #GarageCond
        Paved_drive #PavedDrive
    ],
    handle_unknown = 'use_encoded_value',
    unknown_value = -1
    ))
])

In [ ]:
# ordinal_pipeline = Pipeline(steps = [
#     ('Impute', SimpleImputer(strategy = 'constant', fill_value = 'None')),
#     ('Encoder', OrdinalEncoder(categories = [
#         lShape, #LotShape
#         lSlope, #LotSlope
#         QualOrder, #ExterQual
#         QualOrder, #ExterCond
#         QualOrder, #BsmtQual
#         QualOrder, #BsmtCond
#         bsmt_quality_order, #BsmtExposure
#         bsmt_fin_order, #BsmtFinType1
#         bsmt_fin_order, #BsmtFinType2
#         QualOrder, #HeatingQc
#         Electrical_order, #Electrical
#         Functional_order, #Functional
#         QualOrder, #FireplaceQual
#         Garage_Qual, #GarageFinish
#         QualOrder, #GarageQual
#         QualOrder, #GarageCond
#         Paved_drive #PavedDrive
#     ],
#     handle_unknown = 'use_encoded_value',
#     unknown_value = -1
#     ))
# ])

In [ ]:
nominal_pipeline = Pipeline(steps = [
        ('Impute', SimpleImputer(strategy = 'constant', fill_value = 'None')),
        ('Encoder', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False))
    ]
)

In [ ]:
ordinal_cols = ['LotShape', 'LandSlope', 'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive']

nominal_cols = [c for c in cat_cols if c not in ordinal_cols]
num_cols = [c for c in train.select_dtypes(include = 'number').columns.tolist() if c != 'SalePrice']

preprocessor = ColumnTransformer(transformers = [
    ('num', num_pipeline, num_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols),
    ('nominal', nominal_pipeline, nominal_cols)
])

#### Separating Target and training feature

In [ ]:
X = train.drop(['SalePrice'], axis = 1)
y = train['SalePrice']

In [ ]:
X.head()

In [ ]:
X = preprocessor.fit_transform(X)
test = preprocessor.transform(test)

### Model Training

In [ ]:
models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'RandomForestRegressor': RandomForestRegressor(),
    'AdaBoostRegressor': AdaBoostRegressor(),
    'DecisionTreeRegressor': DecisionTreeRegressor(),
    'XGBRegressor': XGBRegressor(),
    'CatBoostRegressor': CatBoostRegressor(verbose = False)
}

In [ ]:
def evaluate(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return r2, mse, mae

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [ ]:
model_list = []
r2_list = []

for i in range(len(models)):
    model = list(models.values())[i]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2, mse, mae = evaluate(y_test, y_pred)

    print('Model: ', list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Training result: ')
    print('Mean absloute error: ', mae)
    print('Mean squared error: ', mse)
    print('r^2 score: ', r2)

    r2_list.append(r2)

    print('*' * 20)

In [ ]:
result = pd.DataFrame(zip(model_list, r2_list), columns = ['Models', 'Score'])

In [ ]:
result.sort_values(by = 'Score', ascending = False)

### Hyper-Parameter Tuining

In [ ]:
models_params = {
    'Ridge': (Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]}),
    'Lasso': (Lasso(), {'alpha': [0.01, 0.1, 1, 10, 100]}),
    'DecisionTree': (DecisionTreeRegressor(), {
        'max_depth': [3, 5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }),
    'RandomForest': (RandomForestRegressor(), {
        'n_estimators': [100, 200],
        'max_depth': [10, 20, None],
        'max_features': ['sqrt', 'log2']
    }),
    'AdaBoost': (AdaBoostRegressor(), {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    }),
    'XGBoost': (XGBRegressor(), {
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.1],
        'max_depth': [3, 5, 7]
    }),
    'CatBoost': (CatBoostRegressor(verbose=False), {
        'iterations': [100, 200],
        'learning_rate': [0.01, 0.1],
        'depth': [4, 6, 8]
    }),
}

In [ ]:
h_result = {}
for name, (model, params) in models_params.items():
    print('Tuining: ', name)
    grid = RandomizedSearchCV(model, params, scoring = 'r2', n_jobs = -1, cv = 5, n_iter = 20)
    grid.fit(X_train, y_train)

    h_result[name] = {
        'best_estimator': grid.best_estimator_,
        'best_score': grid.best_score_,
        'best_params': grid.best_params_
    }
    print(f'{name} → best R2: {grid.best_score_:.4f} | params: {grid.best_params_}')

### Submission

In [ ]:
best_model_name = result.iloc[-1]['Models']
best_model = models[best_model_name]
final_pipeline = Pipeline(steps = [
    ('best_model', best_model)
    ])

In [ ]:
test_id = pd.read_csv('../data/test.csv')['Id']

test_id = pd.read_csv('../data/test.csv')['Id']

final_pipeline.fit(X, y)

prediction = final_pipeline.predict(test)

submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': prediction
})

submission.to_csv('../data/submission.csv', index = False)